#Tool Use pattern using the LangGraph framework

This example demonstrates a content summarization agent using a tool (summarize_text) and a Gemini LLM to handle instructions intelligently.

✅ Prerequisites


In [3]:
!pip install -q langgraph langchain langchain-google-genai google-generativeai python-dotenv
!pip install -q nest_asyncio

✅ Step-by-Step Jupyter Notebook Code

In [4]:
# Step 1: Imports and setup
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableLambda
from langchain_core.tools import tool
import nest_asyncio
from google.colab import userdata


# Allow nested event loops in Jupyter
nest_asyncio.apply()

# Step 2: Set Google Gemini API Key
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

# Step 3: Initialize Gemini 1.5 Flash Model
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.0, # Lower temperature for more deterministic output
    google_api_key=GOOGLE_API_KEY
)

🧠 Step 4: Define a Tool for Summarization

In [5]:
@tool
def summarize_text(text: str) -> str:
    """Summarize long content into a few key points."""
    return llm.invoke(f"Please summarize this text: {text}")


🔁 Step 5: Define a Function Node to Choose Tool

In [10]:
def tool_router(state: dict) -> dict:
    """Router logic to decide tool usage based on user input."""
    question = state['input']

    if "summarize" in question.lower():
        summary = summarize_text.invoke(input=question)
        return {"output": str(summary)}
    else:
        response = llm.invoke(f"Answer this query: {question}")
        return {"output": str(response)}

🧩 Step 6: Create LangGraph StateGraph

In [11]:
# Define the state
from typing import TypedDict, Annotated, List, Dict

class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        input: User input
        output: Output from the tool or LLM
    """
    input: str
    output: str

graph = StateGraph(GraphState)

# Add a Lambda node to handle logic
graph.add_node("tool_use_node", RunnableLambda(tool_router))

# Set entry point and end
graph.set_entry_point("tool_use_node")
graph.set_finish_point("tool_use_node")

# Compile the graph
runnable_graph = graph.compile()

▶️ Step 7: Run the Graph

In [12]:
# Example input: summarization
user_query = {
    "input": "Can you summarize this? Artificial intelligence is a broad field that includes machine learning, natural language processing, robotics, and many other areas..."
}

result = runnable_graph.invoke(user_query)
print("🔍 Output:", result["output"])


🔍 Output: content='Artificial intelligence (AI) is a wide-ranging field encompassing various subfields like machine learning, natural language processing, and robotics.' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []} id='run--613af600-fa67-46e6-9eb8-d52fb6faa32c-0' usage_metadata={'input_tokens': 32, 'output_tokens': 27, 'total_tokens': 59, 'input_token_details': {'cache_read': 0}}


✅ Output Example

You will see a concise summary or Gemini-generated answer depending on the input.

🧪 Try More Examples

In [13]:
# A direct Q&A (not summarization)
runnable_graph.invoke({"input": "What is the capital of France?"})

# Another summarization
runnable_graph.invoke({"input": "Summarize the theory of relativity in simple words."})


{'input': 'Summarize the theory of relativity in simple words.',
 'output': 'content="Relativity basically says two things:  1) **Space and time are intertwined and relative to the observer\'s motion.**  Faster you move, the slower time passes for you and the shorter distances appear in your direction of travel.  2) **Gravity isn\'t a force, but a curvature of spacetime caused by mass and energy.**  Massive objects warp the fabric of spacetime, causing other objects to move towards them (what we perceive as gravity)." additional_kwargs={} response_metadata={\'prompt_feedback\': {\'block_reason\': 0, \'safety_ratings\': []}, \'finish_reason\': \'STOP\', \'safety_ratings\': []} id=\'run--4e973398-4cdc-464b-9658-a244d9cded5d-0\' usage_metadata={\'input_tokens\': 15, \'output_tokens\': 93, \'total_tokens\': 108, \'input_token_details\': {\'cache_read\': 0}}'}

#🧾 Summary

| Component        | Description                             |
| ---------------- | --------------------------------------- |
| `summarize_text` | Custom tool for summarizing text        |
| `tool_router`    | Decides whether to use LLM or tool      |
| `LangGraph`      | Wraps the logic into a modular workflow |

#✅ Suggested Tools You Can Easily Add

| Tool Name              | Description                                                           | Input Example                        |
| ---------------------- | --------------------------------------------------------------------- | ------------------------------------ |
| `get_weather(city)`    | Returns mock weather for a city (can later connect to real API)       | "What's the weather in Delhi today?" |
| `calculator(expr)`     | Calculates arithmetic expressions                                     | "What's 45 \* (23 - 2)?"             |
| `get_date()`           | Returns the current date and time                                     | "What is the current date?"          |
| `currency_converter()` | Converts currency values (mock version; can be real later)            | "Convert 10 USD to INR"              |
| `wiki_search(topic)`   | Returns a summary of a topic (can use Wikipedia API or Gemini itself) | "Tell me about Quantum Mechanics"    |


#🧪 Example Code for These Tools (Mock Implementations)
You can integrate these easily as @tool functions like below:



In [14]:
from datetime import datetime

@tool
def get_weather(city: str) -> str:
    """Mock weather tool that returns fake weather info for the given city."""
    return f"The weather in {city} is 32°C, mostly sunny."

@tool
def calculator(expr: str) -> str:
    """Evaluates a math expression."""
    try:
        result = eval(expr)
        return f"The result of '{expr}' is {result}"
    except:
        return "Invalid math expression."

@tool
def get_date(_: str = "") -> str:
    """Returns the current date and time."""
    return datetime.now().strftime("Current date and time: %Y-%m-%d %H:%M:%S")

@tool
def currency_converter(_: str = "") -> str:
    """Mock currency conversion tool."""
    return "10 USD is approximately 830 INR (mock conversion)."

@tool
def wiki_search(topic: str) -> str:
    """Mock Wikipedia summary tool using Gemini."""
    return llm.invoke(f"Give me a Wikipedia-style summary of: {topic}")


🧩 Update Tool Router Logic

Update your tool_router() to detect intent and call the appropriate tool:

In [15]:
def tool_router(state: dict) -> dict:
    question = state["input"].lower()

    if "weather" in question:
        city = question.split("in")[-1].strip()
        return {"output": get_weather.invoke(city)}

    elif any(k in question for k in ["calculate", "*", "/", "+", "-"]):
        return {"output": calculator.invoke(question)}

    elif "date" in question:
        return {"output": get_date.invoke("")}

    elif "convert" in question and "usd" in question.lower():
        return {"output": currency_converter.invoke("")}

    elif "summarize" in question:
        return {"output": summarize_text.invoke(text=question)}

    elif "wiki" in question or "tell me about" in question:
        topic = question.replace("tell me about", "").strip()
        return {"output": wiki_search.invoke(topic)}

    else:
        return {"output": llm.invoke(question)}


#✅ Example Prompts You Can Try

>"What's the weather in Mumbai today?"

>"Calculate 256 / 4 + 3"

>"Give me today's date"

>"Convert 50 USD to INR"

>"Tell me about photosynthesis"

>"Summarize this: The Industrial Revolution began in the 18th century..."